### Identify Cell Wall Genes in Non-Arabidopsis Genomes With HMMR3
Grady King

The .xlsx file comes from a literature review in [this paper](https://pmc.ncbi.nlm.nih.gov/articles/PMC10762515/#kiad267-B46) that found all genes in Arabidopsis that relate to cell wall biosynthesis. 

my goal is to build a model that takes these genes, builds a "profile" of how these proteins look, then I can use the model to predict which genes in a different genome fit with the profile, which are likely also related to cell walls. this file is the laziest possible way of doing that, using the off-the-shelf pyhmmer that calls HMMER3, then I can build from there.

In [40]:
import pyhmmer
import pandas as pd
import numpy as np

In [7]:
table_s8 = pd.read_excel("kiad267_supplementary_data.xlsx", sheet_name="Supplemental Table S8", header=1)

In [8]:
print(table_s8)

     Arabidopsis thaliana gene ID (TAIR assembly) Cell wall gene function  \
0                                       At1g05570                    CALS   
1                                       At2g36850                    CALS   
2                                       At4g04970                    CALS   
3                                       At4g03550                    CALS   
4                                       At2g31960                    CALS   
...                                           ...                     ...   
1308                                    At1g62700                    VND5   
1309                                    At5g62380                    VND6   
1310                                    At1g71930                    VND7   
1311                                    At2g44745                  WRKY12   
1312                                    At5g64530                    XND1   

      Arabidopsis thaliana chromosome (TAIR assembly) Start (bp, BC_GFF)  \

The Supplemental Table S8 contains each arabidopsis gene related to cell wall biosynthesis, along with basic information about it, which pfam protein families it belongs to, and its sequence. 

Let's take the pfam values at face value and download all of those profile HMMs. Then, we could say that if a given protein from a new genome matches one of the profiles from arabidopsis, then it must be a cell wall gene. Exactly where the cutoff will be can be decided later.

In [43]:
# isolate pfam IDs so I can download them all

split_pfam = set(table_s8["PFAM (Hmmsearch, E1)"].str.split(';').explode())
split_pfam.remove('')
split_pfam.remove(np.nan)
split_pfam

{'PF00004',
 'PF00005',
 'PF00009',
 'PF00025',
 'PF00046',
 'PF00067',
 'PF00069',
 'PF00071',
 'PF00106',
 'PF00107',
 'PF00141',
 'PF00150',
 'PF00168',
 'PF00169',
 'PF00182',
 'PF00201',
 'PF00221',
 'PF00225',
 'PF00232',
 'PF00235',
 'PF00249',
 'PF00294',
 'PF00295',
 'PF00326',
 'PF00331',
 'PF00332',
 'PF00337',
 'PF00342',
 'PF00349',
 'PF00350',
 'PF00394',
 'PF00408',
 'PF00501',
 'PF00514',
 'PF00534',
 'PF00535',
 'PF00544',
 'PF00553',
 'PF00560',
 'PF00561',
 'PF00642',
 'PF00652',
 'PF00722',
 'PF00735',
 'PF00759',
 'PF00777',
 'PF00847',
 'PF00862',
 'PF00891',
 'PF00892',
 'PF00933',
 'PF00984',
 'PF01031',
 'PF01055',
 'PF01061',
 'PF01073',
 'PF01095',
 'PF01135',
 'PF01190',
 'PF01262',
 'PF01301',
 'PF01357',
 'PF01370',
 'PF01501',
 'PF01565',
 'PF01596',
 'PF01636',
 'PF01697',
 'PF01704',
 'PF01738',
 'PF01762',
 'PF01915',
 'PF01926',
 'PF02018',
 'PF02065',
 'PF02129',
 'PF02140',
 'PF02183',
 'PF02212',
 'PF02263',
 'PF02318',
 'PF02319',
 'PF02364',
 'PF

In [62]:
# download all hmm files using a for loop, requesting from interpro (the database behind pfam)
import requests
import os
import time
from tqdm import tqdm

os.makedirs("hmms", exist_ok = True)

for pfam_id in tqdm(list(split_pfam)):
    if not os.path.exists(f"hmms/{pfam_id}.hmm.gz"):
        with open(f"hmms/{pfam_id}.hmm.gz", "wb+") as f:
            with requests.get(f"https://www.ebi.ac.uk/interpro/api/entry/pfam/{pfam_id}?annotation=hmm") as r:
                f.write(r.content)
        time.sleep(0.25)

100%|██████████| 292/292 [02:25<00:00,  2.01it/s] 


In [90]:
hmms = []
for path in os.listdir('hmms'):
    try:
        with pyhmmer.plan7.HMMFile(f"hmms/{path}", alphabet=pyhmmer.easel.Alphabet.amino()) as hmm:
            hmms.extend(hmm)
    except:
        continue

print(len(hmms), "of", len(os.listdir('hmms')))



gzip: hmms/PF16863.hmm.gz: not in gzip format

gzip: hmms/PF05664.hmm.gz: not in gzip format

gzip: hmms/PF07526.hmm.gz: unexpected end of file

gzip: hmms/PF04048.hmm.gz: not in gzip format

gzip: hmms/PF03360.hmm.gz: unexpected end of file


286 of 297



gzip: hmms/PF02439.hmm.gz: not in gzip format


In [91]:
print(hmms[0].name, hmms[0].alphabet)

AMP-binding AA()


In [93]:
pyhmmer.hmmpress(hmms, 'hmms/pressed')

286

Now, can compare any genome to the families using `hmmscan`

In [ ]:
# example tree genome Fraxinus excelsior obtained from https://www.ncbi.nlm.nih.gov/protein?term=Fraxinus%20excelsior%5BOrganism%5D -> send to: file, FASTA
with pyhmmer.easel.SequenceFile("fraxinus_excelsior.fasta", digital=True) as seq_file:
    sequences = list(seq_file)

len(sequences)

569

In [107]:
with pyhmmer.plan7.HMMPressedFile("hmms/pressed") as hmm_db:
    hits_by_sequence = list(pyhmmer.hmmer.hmmscan(sequences, hmm_db))

print(len(hits_by_sequence))

569


In [ ]:
# how many hits are there total?
sum([len(list(hits)) for hits in hits_by_sequence])

66

In [ ]:
# what is the histogram of how many hits each sequence gets?
pd.Series([len(list(hits)) for hits in hits_by_sequence]).value_counts()

0    539
1     21
5      5
2      2
8      2
Name: count, dtype: int64

In [ ]:
# list out the hits
for hits in hits_by_sequence:
    all_hit = list(hits)
    if len(all_hit) > 0:
        for hit in all_hit:
            print(hits.query.name, hit.name, hit.score, hit.evalue)

XYM53122.1 AAA 39.41424560546875 1.1437173269450989e-11
XYM53102.1 AAA 39.41424560546875 1.1437173269450989e-11
XYM53065.1 AAA 20.949413299560547 5.837232861772399e-06
XYM53065.1 AAA_25 17.184280395507812 4.9065420706025395e-05
XYM53065.1 ABC_tran 16.329483032226562 0.000174095821416802
XYM53065.1 AAA_16 15.70114517211914 0.0002551144943242131
XYM53065.1 RsgA_GTPase 14.241277694702148 0.00044051713144111317
XYM53043.1 AAA_16 13.03866195678711 0.001678505171293055
XVF99162.1 AAA 39.57913589477539 1.0170620810834244e-11
XVF99142.1 AAA 39.57913589477539 1.0170620810834244e-11
XVF99105.1 AAA 20.949413299560547 5.837232861772399e-06
XVF99105.1 AAA_25 17.184280395507812 4.9065420706025395e-05
XVF99105.1 ABC_tran 16.329483032226562 0.000174095821416802
XVF99105.1 AAA_16 15.70114517211914 0.0002551144943242131
XVF99105.1 RsgA_GTPase 14.241277694702148 0.00044051713144111317
XVF99083.1 AAA_16 13.03866195678711 0.001678505171293055
YP_009476360.1 AAA 39.57913589477539 1.0170620810834244e-11
YP_0